In [85]:
import ephem
import numpy as np
from datetime import datetime
from astropy.time import Time
import pandas as pd
import math

In [86]:
def get_moon_parameters(obs_date, lat, lon, elevation=0, pressure=1013.25, horizon='-0:34', epoch='2000'):
    """Calculate Moon and Sun parameters using PyEphem."""
    observer = ephem.Observer()
    observer.lat = str(lat)
    observer.lon = str(lon)
    observer.elev = elevation  # Elevation in meters
    observer.pressure = pressure  # Atmospheric pressure in millibars
    observer.horizon = horizon  # Default horizon accounting for refraction
    observer.epoch = epoch  # Reference epoch for calculations
#     observer.date = obs_date.datetime  # Direct datetime assignment
    observer.date = obs_date

    
    # Ensure date is in UTC format and properly assigned
#     observer.date = ephem.Date(obs_date)  # Convert to Ephem Date format
    
    moon = ephem.Moon()
    sun = ephem.Sun()
    
    # Compute positions for the observer's location
    moon.compute(observer)
    sun.compute(observer)

    moon.compute(observer)

    for attr in dir(moon):
        if not attr.startswith("_"):
            try:
                val = getattr(moon, attr)
                if not callable(val):
                    print(attr, val)
            except Exception:
                pass
    

    # Convert values from radians to degrees
    moon_phase = moon.phase  # Percentage of moon illuminated
    moon_alt = np.degrees(float(moon.alt))  # Convert altitude from radians to degrees
    moon_az = np.degrees(float(moon.az))  # Convert azimuth from radians to degrees
    sun_alt = np.degrees(float(sun.alt))  # Convert altitude from radians to degrees
    sun_az = np.degrees(float(sun.az))  # Convert azimuth from radians to degrees
    
    arcv = moon_alt - sun_alt  # Geocentric altitude difference
    moon_illum = moon_phase  # Same as phase in percentage
    moon_ang_diam = moon.size / 60.0  # Convert arcseconds to arcminutes
    moon_dist = moon.earth_distance  # Distance in AU
    elongation = np.degrees(ephem.separation((moon.g_ra, moon.g_dec),(sun.g_ra, sun.g_dec)))  # Moon-Sun elongation in degrees
    daz = sun_az - moon_az  # Ensures DAZ is between 0° and 180°

    # Compute ARCL (Arc of Light)
    arcl = elongation  # In degrees, as per Yallop's definition
       
   # Compute SD' using SD' = SD * (1 + sin h * sin π)
    parallax = np.arcsin(6378.1 / (moon_dist * 149597870.7))  # Earth's radius / Moon distance in km
    sd = 0.27245 * np.sin(parallax)  # Semi-diameter in arcminutes
    parallax = np.arcsin(6378.1 / (moon_dist * 149597870.7))  # Earth's radius / Moon distance in km
    sd_prime = sd * (1 + np.sin(np.radians(moon_alt)) * np.sin(parallax))
    
      # Compute W (Crescent Width) using W = SD' * (1 - cos(ARCL))
    w = sd_prime * (1 - np.cos(np.radians(arcl)))
        
    try:
        sunset = observer.next_setting(sun).datetime().strftime('%Y/%m/%d %H:%M:%S')
    except ephem.AlwaysUpError:
        sunset = "Sun always up"
    except ephem.NeverUpError:
        sunset = "Sun never rises"
    
    try:
        moonset = observer.next_setting(moon).datetime().strftime('%Y/%m/%d %H:%M:%S')
    except ephem.AlwaysUpError:
        moonset = "Moon always up"
    except ephem.NeverUpError:
        moonset = "Moon never rises"
    
    return {
        "moon_altitude": moon_alt,
        "moon_azimuth": moon_az,
        "sun_altitude": sun_alt,
        "sun_azimuth": sun_az,
        "arcv": arcv,
        "arcl": arcl,
        "crescent_width": w,
        "moon_phase": moon_phase,
        "moon_illumination": moon_illum,
        "moon_angular_diameter": moon_ang_diam,
        "moon_distance_au": moon_dist,
        "daz": daz,
        "elongation": elongation,
        "sunset_time": sunset,
        "moonset_time": moonset
    }

In [ ]:
import datetime
import pytz
tz = pytz.timezone("Asia/Karachi")
local_date = datetime.datetime.now(tz)
# Convert to UTC for PyEphem
moon_data = get_moon_parameters(local_date, 24.86, 66.99)

a_dec -2:01:54.9
a_epoch 2000/1/1 00:00:00
a_ra 11:58:21.54
alt -9:04:01.2
az 88:37:19.0
circumpolar False
colong 47:11:34.5
dec -2:33:43.7
earth_distance 0.0026306689251214266
elong 142:35:26.6
g_dec -2:10:44.8
g_ra 11:59:42.82
ha 17:24:46.08
hlat -2:01:39.7
hlon 180:47:59.4
hlong 180:47:59.4
libration_lat 2:43:43.8
libration_long 5:17:16.9
mag -11.66
moon_phase 0.8973425505696965
name Moon
neverup False
phase 89.76119995117188
ra 12:03:03.40
radius 0:15:12.0
rise_az 92:24:14.5
rise_time 2026/4/28 11:09:54
set_az 264:18:23.2
set_time 2026/4/28 23:19:20
size 1823.955322265625
subsolar_lat 1:22:14.6
sun_distance 1.0088205337524414
transit_alt 60:51:36.8
transit_time 2026/4/28 17:17:28
